# MiniGenie — Colab Setup Template

Base template for all MiniGenie Colab training sessions.  
Run these cells at the start of **every** session to mount Drive, sync code, install deps, and verify GPU.

**Do not run training in this notebook** — use the dedicated training notebooks:
- `01_train_vqvae.ipynb` — VQ-VAE tokenizer training
- `02_train_dynamics.ipynb` — Flow matching dynamics model training

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Project root on Drive (checkpoints + data persist here across sessions)
DRIVE_PROJECT = '/content/drive/MyDrive/minigenie'

import os
os.makedirs(DRIVE_PROJECT, exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/checkpoints/vqvae', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/checkpoints/dynamics', exist_ok=True)
print(f'Drive project root: {DRIVE_PROJECT}')
print('Contents:', os.listdir(DRIVE_PROJECT))

## 2. Clone / Sync Code from GitHub

In [ ]:
REPO_URL = 'https://github.com/BrutalCaeser/minigenie.git'  # <-- UPDATE THIS
LOCAL_CODE = '/content/minigenie'

if os.path.exists(LOCAL_CODE):
    !cd {LOCAL_CODE} && git pull --ff-only
else:
    !git clone {REPO_URL} {LOCAL_CODE}

os.chdir(LOCAL_CODE)
!git log --oneline -5
print(f'\nWorking directory: {os.getcwd()}')

## 3. Install Dependencies

In [ ]:
# Install project dependencies (Colab already has PyTorch + CUDA)
!pip install -q einops pyyaml wandb tqdm imageio scipy pillow matplotlib torchvision

# Install project in editable mode so 'from src.models import ...' works
!pip install -q -e .

# Verify imports
import torch
import einops
import yaml
print(f'PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
print('einops OK, pyyaml OK')

## 4. Verify GPU

In [ ]:
assert torch.cuda.is_available(), 'No GPU detected! Go to Runtime > Change runtime type > GPU'

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f'GPU: {gpu_name}')
print(f'VRAM: {gpu_mem:.1f} GB')

# Quick sanity check
x = torch.randn(2, 3, 64, 64, device='cuda')
print(f'Tensor on GPU: {x.shape}, dtype={x.dtype}, device={x.device}')
del x
torch.cuda.empty_cache()
print('\nGPU verified and ready!')

## 5. Extract Data (if needed)

Upload `coinrun_data.tar.gz` to Google Drive at `MyDrive/minigenie/` before running this.

In [ ]:
DATA_TAR = f'{DRIVE_PROJECT}/coinrun_data.tar.gz'
LOCAL_DATA = '/content/minigenie/data/coinrun/episodes'

if os.path.exists(LOCAL_DATA) and len(os.listdir(LOCAL_DATA)) > 100:
    n_eps = len([f for f in os.listdir(LOCAL_DATA) if f.endswith('.npz') and not f.startswith('._')])
    print(f'Data already extracted: {n_eps} episodes in {LOCAL_DATA}')
elif os.path.exists(DATA_TAR):
    print(f'Extracting {DATA_TAR} ...')
    !tar xzf {DATA_TAR} -C /content/minigenie/
    n_eps = len([f for f in os.listdir(LOCAL_DATA) if f.endswith('.npz') and not f.startswith('._')])
    print(f'Extracted {n_eps} episodes')
else:
    print(f'Data archive not found at {DATA_TAR}')
    print('Upload coinrun_data.tar.gz to Google Drive at MyDrive/minigenie/')

## 6. Verify Data Integrity

In [ ]:
import numpy as np
import glob

data_dir = '/content/minigenie/data/coinrun/episodes'
paths = sorted([p for p in glob.glob(f'{data_dir}/*.npz') if not os.path.basename(p).startswith('._')])
print(f'Found {len(paths)} episodes')

# Spot-check a few episodes
for i in [0, len(paths)//2, -1]:
    ep = np.load(paths[i])
    frames = ep['frames']
    actions = ep['actions']
    print(f'  {os.path.basename(paths[i])}: frames {frames.shape} {frames.dtype}, '
          f'actions {actions.shape} range [{actions.min()}, {actions.max()}]')

assert len(paths) >= 1000, f'Expected >=1000 episodes, got {len(paths)}'
print('\nData integrity check passed!')

---

## Setup Complete

Everything is ready. Switch to the appropriate training notebook:
- **VQ-VAE training:** `01_train_vqvae.ipynb`
- **Dynamics training:** `02_train_dynamics.ipynb`

Or continue below to run the smoke test as a final verification.

In [ ]:
# Optional: run smoke test to verify everything works end-to-end
!cd /content/minigenie && python scripts/smoke_test.py